In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold


In [ ]:
dataset = pd.read_csv('Wine_Test_01.csv') # This line of code reads the dataset from csv file

### data is split into features and output ### 
features_pd = dataset.drop('quality', axis=1) 
output_pd = dataset['quality']

### data is converted to numpy arrays because sklearn works with numpy arrays ###
features = np.array(features_pd)
quality = np.array(output_pd)

In [ ]:
## create an estimator for the RFE selector since RFE needs a supervised learning model for prediction ##
estimator = SVC(kernel='linear', C=1000, gamma=0.1)

## create the RFE selector ##
## n_features_to_select is the number of features to select ##
selector = RFE(estimator=estimator, n_features_to_select=8) 

## split the data into train and test sets with 80% train and 20% test ##
X_train, X_test, y_train, y_test = train_test_split(features, quality, test_size=0.2, random_state=42)

## fit the RFE selector to the training data ##
selector.fit(X_train, y_train)


,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).","SVC(C=1000, g...rnel='linear')"
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",8
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1000
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True


In [ ]:
'''
after fitting the RFE selector to the training data, 
we can print the number of features selected and the features that are selected
so in the next loop we can print the features that are selected
'''
for i, x in enumerate(selector.support_):
    if x:
        print("the feature {} is important".format(features_pd.columns[i]))
        
## the next 2 lines of code are used to transform the training and test data to the selected features ##
## the features that are not selected are removed ##
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

## train the model on the selected features ##
estimator.fit(X_train_selected, y_train)

## predict the output on the test data with the modell fitted to the selected features ##
y_pred = estimator.predict(X_test_selected)

## print the accuracy of the model ##
print("Accuracy: ", np.mean(y_pred == y_test))

the feature fixed acidity is important
the feature volatile acidity is important
the feature citric acid is important
the feature chlorides is important
the feature density is important
the feature pH is important
the feature sulphates is important
the feature alcohol is important
(914, 12)
(914, 12)
Accuracy:  0.7510917030567685


In [ ]:
## The following function executes the instructions of task 1c) ##

def grid_search(X):
    
    model = SVC() # the model that will be used for grid search
    
    ## the hyperparemeters that will be used for grid search ##
    hyperparemeters = {'C': [0.1, 1, 10, 100, 1000], 
                    'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
                    'kernel': ['linear', 'rbf']}

    ## a list to store the accuracy of each iteration ##    
    accuracy = []

    ## loop over 10 iterations of modeling and testing ##
    for i in range(10):
        ## New random train and test sets are created for each iteration ##
        X_train, X_test, y_train, y_test = train_test_split(X, quality, test_size=0.2, random_state=i)
        
        ## GridSearchCV is used to find the best hyperparemeters for the model ##
        grid_search = GridSearchCV(model, hyperparemeters, cv=10, refit=True, verbose=0)
        
        grid_search.fit(X_train, y_train)
        
        ## The accuracy of the model with the best hyperparemeters is retrieved then stored in the accuracy list ##
        acc_score = grid_search.best_estimator_.score(X_test, y_test)
        accuracy.append(acc_score)
        
        ## print out the accuracy and parameter of the best model ##
        print("iteration", i, "Accuracy: ", acc_score, "Best params: ", grid_search.best_params_)

    ## convert the accuracy list to a numpy array to facilitate calculations ##
    accuracy = np.array(accuracy)
    
    ## print out the mean and standard deviation of the accuracy ##
    print("Accuracy: ", np.mean(accuracy), "+/-", np.std(accuracy))

In [ ]:
X_selected = selector.transform(features) # this line of code remove the features that are not selected previously by the RFE selector,
# The RFE is applied on the whole dataset.
grid_search(X_selected) # execute instructions from task 1c) with the selected features only.

iteration 0 Accuracy:  0.7860262008733624 Best params:  {'C': 100, 'gamma': 1, 'kernel': 'rbf'}
iteration 1 Accuracy:  0.7467248908296943 Best params:  {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
iteration 2 Accuracy:  0.7117903930131004 Best params:  {'C': 1000, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 3 Accuracy:  0.7554585152838428 Best params:  {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
iteration 4 Accuracy:  0.74235807860262 Best params:  {'C': 1000, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 5 Accuracy:  0.7379912663755459 Best params:  {'C': 100, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 6 Accuracy:  0.7292576419213974 Best params:  {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
iteration 7 Accuracy:  0.7205240174672489 Best params:  {'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
iteration 8 Accuracy:  0.7510917030567685 Best params:  {'C': 10, 'gamma': 1, 'kernel': 'linear'}
iteration 9 Accuracy:  0.7205240174672489 Best params:  {'C': 1000, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy:  0.7

In [ ]:
auto_estimator = SVC(kernel='linear', C=100, gamma=0.1) # This is an estimator for the RFECV selector

## The RFECV selects both the number of features to select and the features to select ##
## cv parameter is used to specify the number of folds for cross-validation ##
## scoring parameter is used to specify the scoring metric for cross-validation ##
## step parameter is used to specify the number of features to remove at each iteration ##
auto_selector = RFECV(estimator=auto_estimator, step=2, 
                    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                    scoring='accuracy')

auto_selector.fit(X_train, y_train)



(914, 12)


,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,"SVC(C=100, ga...rnel='linear')"
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",5
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",'accuracy'
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",100
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the 

In [ ]:
print("Optimal number of features: %d" % auto_selector.n_features_)

## print the features that are selected ##
for i, x in enumerate(selector.support_):
    if x:
        print("the feature {} is important".format(features_pd.columns[i]))
        
## the next 2 lines of code are used to transform the training and test data to the selected features ##
X_train_auto_selected = auto_selector.transform(X_train)
X_test_auto_selected = auto_selector.transform(X_test)

auto_estimator.fit(X_train_auto_selected, y_train)

y_auto_pred = auto_estimator.predict(X_test_auto_selected)

print("Accuracy: ", np.mean(y_auto_pred == y_test))

Optimal number of features: 7
the feature fixed acidity is important
the feature volatile acidity is important
the feature citric acid is important
the feature chlorides is important
the feature density is important
the feature pH is important
the feature sulphates is important
the feature alcohol is important
Accuracy:  0.7336244541484717


In [ ]:
X_auto_selected = auto_selector.transform(features)

grid_search(X_auto_selected)

iteration 0 Accuracy:  0.7816593886462883 Best params:  {'C': 1000, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 1 Accuracy:  0.7510917030567685 Best params:  {'C': 10, 'gamma': 1, 'kernel': 'rbf'}
iteration 2 Accuracy:  0.7248908296943232 Best params:  {'C': 10, 'gamma': 1, 'kernel': 'rbf'}
iteration 3 Accuracy:  0.7248908296943232 Best params:  {'C': 100, 'gamma': 1, 'kernel': 'rbf'}
iteration 4 Accuracy:  0.759825327510917 Best params:  {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
iteration 5 Accuracy:  0.7248908296943232 Best params:  {'C': 1000, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 6 Accuracy:  0.7161572052401747 Best params:  {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
iteration 7 Accuracy:  0.7467248908296943 Best params:  {'C': 10, 'gamma': 1, 'kernel': 'rbf'}
iteration 8 Accuracy:  0.759825327510917 Best params:  {'C': 1000, 'gamma': 0.01, 'kernel': 'rbf'}
iteration 9 Accuracy:  0.7292576419213974 Best params:  {'C': 1000, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy:  0.74192139737991